# Phase 1 – Task 3: EDA & Preprocessing
**Tindihan ni BERB (E-Commerce)** · Dataset: UCI Online Retail · CSE315 Data Science · BSCS 3A

## 0. Setup & Load Data
Run the download cell, or upload `Online Retail.xlsx` manually to Colab (Files panel on the left) and skip the download lines.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:,.2f}".format)

# Option A: download directly from UCI
!wget -q -O online_retail.zip https://archive.ics.uci.edu/static/public/352/online+retail.zip
!unzip -o -q online_retail.zip

# Option B: if the download fails, upload 'Online Retail.xlsx' manually and delete the 2 lines above
df_raw = pd.read_excel("Online Retail.xlsx")
df = df_raw.copy()
print(df.shape)
df.head()

## 1. Exploratory Data Analysis (Raw Data)

In [ ]:
df.info()
print()
print(df.describe(include="all").T)

In [ ]:
# Missing values
miss = pd.DataFrame({"missing": df.isna().sum(), "percent": df.isna().mean()*100})
print(miss)

# Duplicates
print("\nDuplicate rows:", df.duplicated().sum())

# Cancellations, negative quantity, zero/negative price
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
print("Cancelled rows (InvoiceNo starts with 'C'):", df["InvoiceNo"].str.startswith("C").sum())
print("Rows with Quantity <= 0:", (df["Quantity"] <= 0).sum())
print("Rows with UnitPrice <= 0:", (df["UnitPrice"] <= 0).sum())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(df.isna(), cbar=False, yticklabels=False, ax=ax[0]); ax[0].set_title("Missing values map")
df["Country"].value_counts().head(10).plot.barh(ax=ax[1]); ax[1].invert_yaxis(); ax[1].set_title("Top 10 countries by rows")
plt.tight_layout(); plt.show()

## 2. Data Cleaning
Steps: (a) keep cancellations in a separate table for research question 4, (b) remove duplicates, (c) drop rows with no CustomerID, (d) remove invalid quantity/price, (e) remove non-product codes.

In [ ]:
log = {"raw rows": len(df)}

# (a) Separate cancellations (needed for Research Question 4)
cancelled = df[df["InvoiceNo"].str.startswith("C")].copy()
log["cancellation rows (saved separately)"] = len(cancelled)
df = df[~df["InvoiceNo"].str.startswith("C")]

# (b) Duplicates
before = len(df); df = df.drop_duplicates(); log["duplicates removed"] = before - len(df)

# (c) Missing CustomerID (cannot be used for customer segmentation)
before = len(df); df = df.dropna(subset=["CustomerID"]); log["missing CustomerID removed"] = before - len(df)

# (d) Invalid quantity / price
before = len(df); df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]; log["Quantity/UnitPrice <= 0 removed"] = before - len(df)

# (e) Non-product StockCodes (postage, fees, bank charges, samples, etc.)
df["StockCode"] = df["StockCode"].astype(str).str.upper()
non_product = ["POST", "D", "M", "BANK CHARGES", "PADS", "DOT", "CRUK", "S", "AMAZONFEE", "B"]
before = len(df); df = df[~df["StockCode"].isin(non_product)]; log["non-product codes removed"] = before - len(df)

# Messy Description text
df["Description"] = df["Description"].astype(str).str.strip().str.upper()

df["CustomerID"] = df["CustomerID"].astype(int).astype(str)
log["clean rows"] = len(df)
pd.Series(log, name="rows")

**Outliers:** check Quantity and UnitPrice using the IQR rule. Many buyers are resellers who buy in bulk, so large quantities can be genuine. We cap (winsorize) extreme values instead of deleting them.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
sns.boxplot(x=df["Quantity"], ax=ax[0]); ax[0].set_title("Quantity (before capping)")
sns.boxplot(x=df["UnitPrice"], ax=ax[1]); ax[1].set_title("UnitPrice (before capping)")
plt.tight_layout(); plt.show()

for col in ["Quantity", "UnitPrice"]:
    q1, q3 = df[col].quantile([0.25, 0.75]); iqr = q3 - q1
    n_out = ((df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)).sum()
    print(f"{col}: {n_out:,} IQR outliers ({n_out/len(df):.1%})")

# Cap at 1st and 99th percentile (original columns are kept)
for col in ["Quantity", "UnitPrice"]:
    lo, hi = df[col].quantile([0.01, 0.99])
    df[col + "_capped"] = df[col].clip(lo, hi)

## 3. Transformation & Feature Engineering

In [ ]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]
df["Year"]  = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["YearMonth"] = df["InvoiceDate"].dt.to_period("M").astype(str)
df["DayOfWeek"] = df["InvoiceDate"].dt.day_name()
df["Hour"] = df["InvoiceDate"].dt.hour
df[["InvoiceNo","Description","Quantity","UnitPrice","TotalPrice","YearMonth"]].head()

## 4. Data Integration
We build three analysis-ready tables, one per research question (RQ1 basket, RQ2 customers, RQ4 cancellations).

In [ ]:
# (1) Basket table for Association Rules (RQ1): invoice x product, True/False
top_products = df["Description"].value_counts().head(100).index   # keeps the matrix manageable
basket = (df[df["Description"].isin(top_products)]
          .groupby(["InvoiceNo", "Description"])["Quantity"].sum()
          .unstack(fill_value=0))
basket = basket > 0
print("Basket matrix:", basket.shape)

# (2) Customer table with RFM features for Clustering (RQ2)
snapshot = df["InvoiceDate"].max() + pd.Timedelta(days=1)
rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("TotalPrice", "sum"),
    AvgBasketSize=("Quantity", "mean"),
    UniqueProducts=("StockCode", "nunique"),
).reset_index()
print("Customer table:", rfm.shape)

# (3) Cancellations table (RQ4)
cancelled["Quantity"] = cancelled["Quantity"].abs()
cancelled["Description"] = cancelled["Description"].astype(str).str.strip().str.upper()
print("Cancellation table:", cancelled.shape)
rfm.head()

## 5. Data Reduction
Reduce products (top 100 for the basket) and check customer feature correlation, then log-transform and scale (needed for k-Means later).

In [ ]:
plt.figure(figsize=(6, 4.5))
sns.heatmap(rfm.drop(columns="CustomerID").corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation of customer features"); plt.show()

from sklearn.preprocessing import StandardScaler
rfm_log = rfm.copy()
for c in ["Recency","Frequency","Monetary","AvgBasketSize","UniqueProducts"]:
    rfm_log[c] = np.log1p(rfm_log[c])
features = ["Recency","Frequency","Monetary"]
rfm_scaled = pd.DataFrame(StandardScaler().fit_transform(rfm_log[features]), columns=features)
rfm_scaled.insert(0, "CustomerID", rfm["CustomerID"])
rfm_scaled.describe().loc[["mean","std"]]

## 6. EDA on Clean Data (Research Questions 3 and 4)

In [ ]:
# RQ3: monthly sales
monthly = df.groupby("YearMonth")["TotalPrice"].sum()
plt.figure(figsize=(11, 4)); monthly.plot(marker="o"); plt.title("Monthly revenue"); plt.ylabel("Revenue"); plt.xticks(rotation=45); plt.show()

# RQ3: top 10 products by revenue and by quantity
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
df.groupby("Description")["TotalPrice"].sum().nlargest(10).sort_values().plot.barh(ax=ax[0], title="Top 10 products by revenue")
df.groupby("Description")["Quantity"].sum().nlargest(10).sort_values().plot.barh(ax=ax[1], title="Top 10 products by quantity")
plt.tight_layout(); plt.show()

# Orders by weekday and hour
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
df.groupby("DayOfWeek")["InvoiceNo"].nunique().reindex(order).plot.bar(ax=ax[0], title="Orders per weekday")
df.groupby("Hour")["InvoiceNo"].nunique().plot.bar(ax=ax[1], title="Orders per hour")
plt.tight_layout(); plt.show()

# RQ4: cancellation rate and most-cancelled products
total_invoices = df["InvoiceNo"].nunique() + cancelled["InvoiceNo"].nunique()
print(f"Cancelled invoices: {cancelled['InvoiceNo'].nunique():,} of {total_invoices:,} "
      f"({cancelled['InvoiceNo'].nunique()/total_invoices:.1%})")
cancelled.groupby("Description")["Quantity"].sum().nlargest(10).sort_values().plot.barh(
    figsize=(8,4), title="Top 10 most cancelled products (units)"); plt.show()

# Customer spending distribution
sns.histplot(rfm["Monetary"], bins=50, log_scale=True); plt.title("Customer total spend (log scale)"); plt.show()

## 7. Save Outputs (for Phase 2)

In [ ]:
df.to_csv("clean_transactions.csv", index=False)
rfm.to_csv("customer_rfm.csv", index=False)
rfm_scaled.to_csv("customer_rfm_scaled.csv", index=False)
basket.to_csv("basket_matrix.csv")
cancelled.to_csv("cancellations.csv", index=False)

print(pd.Series(log, name="rows"))   # copy this table into the report
print("\nFinal clean shape:", df.shape)